# Application entry point

> Startup orchestration for the monitoring controller, previous-session report, and system-tray interface.

This module provides the application’s top-level startup function. It creates the monitoring controller, begins a monitoring session, opens the report for the last completed session, and then starts the tray interface.

When the exported module is run directly, it calls `main()` to launch the application. Lifecycle and monitoring behaviour remain delegated to `MonitorController` and the tray module.

In [ ]:
#| hide
from nbdev.showdoc import *

## Application dependencies

In [ ]:
#| default_exp main

In [ ]:
#| exporti
import ctypes
import logging
import sys
from ctypes import wintypes
from logging.handlers import RotatingFileHandler
from platformdirs import user_log_path

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| exporti
import snooper_pkg.config as cf
from snooper_pkg.monitoring_controller import MonitorController
from snooper_pkg.tray import *

ModuleNotFoundError: No module named 'win32gui'

In [ ]:
#| exporti
LOG_DIR = user_log_path(cf.APP_NAME, appauthor=False)
LOG_PATH = LOG_DIR / cf.LOG_FILENAME

NameError: name 'cf' is not defined

In [ ]:
#| exporti
ERROR_ALREADY_EXISTS = 183

_MUTEX_NAME = (
    r"Local\SnooperPkg.SingleInstance."
    r"7E241C35-DBF7-4E4B-89F3-3B8A278D1E8B"
)

if sys.platform == "win32":
    _kernel32 = ctypes.WinDLL("kernel32", use_last_error=True)

    _kernel32.CreateMutexW.argtypes = (
        ctypes.c_void_p,
        wintypes.BOOL,
        wintypes.LPCWSTR,
    )
    _kernel32.CreateMutexW.restype = wintypes.HANDLE

    _kernel32.CloseHandle.argtypes = (wintypes.HANDLE,)
    _kernel32.CloseHandle.restype = wintypes.BOOL
else:
    _kernel32 = None

In [ ]:
#| exporti
def _acquire_single_instance():
    "Return the mutex handle and whether this is the primary instance."
    if _kernel32 is None:
        return None, True

    ctypes.set_last_error(0)
    handle = _kernel32.CreateMutexW(None, False, _MUTEX_NAME)
    error = ctypes.get_last_error()

    if not handle:
        raise ctypes.WinError(error)

    if error == ERROR_ALREADY_EXISTS:
        _kernel32.CloseHandle(handle)
        return None, False

    return handle, True


In [ ]:
#| exporti
def configure_logging():
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

    handler = RotatingFileHandler(
        LOG_PATH,
        maxBytes=1_000_000,
        backupCount=3,
        encoding="utf-8",
    )

    logging.basicConfig(
        level=getattr(logging, cf.LOG_LEVEL.upper()),
        format="%(asctime)s %(levelname)s %(name)s: %(message)s",
        handlers=[handler],
    )

## Application startup

In [ ]:
#| export
def main():
    "Start Snooper unless another instance is already running."
    mutex, is_primary = _acquire_single_instance()

    if not is_primary:
        return

    try:
        configure_logging()

        controller = MonitorController()
        controller.start_monitoring()

        run_tray(controller, show_last_on_start=True)
    finally:
        if mutex is not None:
            _kernel32.CloseHandle(mutex)


In [ ]:
#| exporti
#| eval: false
if __name__ == "__main__":
    main()

NameError: name 'main' is not defined

- `snooper_pkg.config` is imported as `cf` but is not used anywhere in the current notebook.
- The wildcard tray import hides which tray APIs this module depends on. The current code uses `open_last_session_report` and `run_tray`; changing the import would affect executable code and is therefore not included here.
- The exported `if __name__ == "__main__":` cell may call `main()` when executed interactively because notebook kernels commonly set `__name__` to `"__main__"`. Confirm that launching the application during a notebook “Run All” is intended.
- Importing `MonitorController` transitively imports Windows-only modules such as `win32gui`. The notebook therefore cannot currently be executed in this non-Windows environment.
- Existing cell outputs show earlier import statements and failures that no longer exactly match the current source cells. They should eventually be cleared or regenerated during the real Windows test, but no output changes were made here.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()